In [1]:
# PYSPARK_PYTHON :- tells spark which python interpreter to use on worker node
# PYSPARK_DRIVER_PYTHON :- tells spark which python interpreter to use for the driver program
import os
os.environ["PYSPARK_PYTHON"] = "D:\AllSubjectNotes\WEB_TECH_NOTES\DATA_ENGINEER\DAY4\pyspark_practice\venv\scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "D:\AllSubjectNotes\WEB_TECH_NOTES\DATA_ENGINEER\DAY4\pyspark_practice\venv\scripts\python.exe"

Questions :Realted to pyspark CSV Source :- https://www.kaggle.com/datasets/urvishahir/electric-vehicle-specifications-dataset-2025


In [2]:
# SetUp


# %pip install pyspark


import pyspark.sql as ps
import pyspark.sql.functions as psf
import pyspark.sql.types as pst
import pandas as pd
import pyspark.sql.window as psw

In [3]:
# create a spark object
# sparkSession :- main entry point working with pyspark
# builder :- starts building / configuration 
# appName :- set the name of the spark application 
# getOrCreate :- get an existing sparl object or create a new spark object


spark = (ps.SparkSession.builder.appName("EV Example").getOrCreate())

In [4]:
file_path = "electric_vehicles_spec_2025.csv.csv"

In [5]:
df = (
    spark
    .read
    .option("samplingRatio",0.001) # we are taking the data for the sampling purpose 0.1%
    .option("header","true") # treats  the first row of the csv file as column name 
    .option("inferSchema","true") # detects the datatype
    .csv(file_path)
)

In [6]:
df.columns

['brand',
 'model',
 'top_speed_kmh',
 'battery_capacity_kWh',
 'battery_type',
 'number_of_cells',
 'torque_nm',
 'efficiency_wh_per_km',
 'range_km',
 'acceleration_0_100_s',
 'fast_charging_power_kw_dc',
 'fast_charge_port',
 'towing_capacity_kg',
 'cargo_volume_l',
 'seats',
 'drivetrain',
 'segment',
 'length_mm',
 'width_mm',
 'height_mm',
 'car_body_type',
 'source_url']

In [7]:
(df.createOrReplaceTempView("df_view"))


1. DataFrame Basics :- 

1. Load the csv file into a dataframe and print the schema.

In [8]:
df = (spark.read.option("samplingRatio",0.001).option("header","true").option("inferSchema","true").csv(file_path))

# df.columns
df.printSchema()

root
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- top_speed_kmh: integer (nullable = true)
 |-- battery_capacity_kWh: double (nullable = true)
 |-- battery_type: string (nullable = true)
 |-- number_of_cells: integer (nullable = true)
 |-- torque_nm: integer (nullable = true)
 |-- efficiency_wh_per_km: integer (nullable = true)
 |-- range_km: integer (nullable = true)
 |-- acceleration_0_100_s: double (nullable = true)
 |-- fast_charging_power_kw_dc: integer (nullable = true)
 |-- fast_charge_port: string (nullable = true)
 |-- towing_capacity_kg: integer (nullable = true)
 |-- cargo_volume_l: integer (nullable = true)
 |-- seats: integer (nullable = true)
 |-- drivetrain: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- length_mm: integer (nullable = true)
 |-- width_mm: integer (nullable = true)
 |-- height_mm: integer (nullable = true)
 |-- car_body_type: string (nullable = true)
 |-- source_url: string (nullable = true)



2. Display the first 10 rows of the dataframe.

In [9]:
df.show(10)

+------+--------------------+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+------------+---------+--------+---------+-------------+--------------------+
| brand|               model|top_speed_kmh|battery_capacity_kWh|battery_type|number_of_cells|torque_nm|efficiency_wh_per_km|range_km|acceleration_0_100_s|fast_charging_power_kw_dc|fast_charge_port|towing_capacity_kg|cargo_volume_l|seats|drivetrain|     segment|length_mm|width_mm|height_mm|car_body_type|          source_url|
+------+--------------------+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+------------+---------+--------+---------+-------------+--------------------+
|Abarth|    500e Conve

In [10]:
# 3 Select  only brand , model and range_km columns.

ans_3rd = (df.select("brand","model","range_km"))
ans_3rd.show(10)

+------+--------------------+--------+
| brand|               model|range_km|
+------+--------------------+--------+
|Abarth|    500e Convertible|     225|
|Abarth|      500e Hatchback|     225|
|Abarth| 600e Scorpionissima|     280|
|Abarth|        600e Turismo|     280|
|Aiways|                  U5|     315|
|Aiways|                  U6|     350|
|  Alfa|Romeo Junior Elet...|     320|
|  Alfa|Romeo Junior Elet...|     310|
|Alpine|A290 Electric 180 hp|     310|
|Alpine|A290 Electric 220 hp|     305|
+------+--------------------+--------+
only showing top 10 rows



4. Rename the columns battery_capacity_kwh to battery_kwh.

In [11]:
df_renamed = (df.withColumnRenamed("battery_capacity_kwh","battery_kwh"))
df_renamed.columns

['brand',
 'model',
 'top_speed_kmh',
 'battery_kwh',
 'battery_type',
 'number_of_cells',
 'torque_nm',
 'efficiency_wh_per_km',
 'range_km',
 'acceleration_0_100_s',
 'fast_charging_power_kw_dc',
 'fast_charge_port',
 'towing_capacity_kg',
 'cargo_volume_l',
 'seats',
 'drivetrain',
 'segment',
 'length_mm',
 'width_mm',
 'height_mm',
 'car_body_type',
 'source_url']

5. Drop the source_url column from the dataframe.

In [12]:
len(df.columns)

22

In [13]:
drop_source_url = (df.drop("source_url"))
len(drop_source_url.columns)

21

# Filtering & Selection

6. Filter rows where brand name is "Abrath"

In [14]:
(
    df.filter(psf.expr("brand='Abarth'"))
    .select("brand","model")
    .show()
)

+------+-------------------+
| brand|              model|
+------+-------------------+
|Abarth|   500e Convertible|
|Abarth|     500e Hatchback|
|Abarth|600e Scorpionissima|
|Abarth|       600e Turismo|
+------+-------------------+



7. Find all vehicle with top_speed_kmh greater than 180.

In [15]:
query = """
     select brand , model , top_speed_kmh
     from df_view
     where top_speed_kmh > 180 
"""

In [16]:
spark.sql(query).show(10)

+------+--------------------+-------------+
| brand|               model|top_speed_kmh|
+------+--------------------+-------------+
|Abarth| 600e Scorpionissima|          200|
|Abarth|        600e Turismo|          200|
|  Alfa|Romeo Junior Elet...|          200|
|  Audi|     A6 Avant e-tron|          210|
|  Audi|A6 Avant e-tron p...|          210|
|  Audi|A6 Avant e-tron q...|          210|
|  Audi| A6 Sportback e-tron|          210|
|  Audi|A6 Sportback e-tr...|          210|
|  Audi|A6 Sportback e-tr...|          210|
|  Audi|           Q6 e-tron|          210|
+------+--------------------+-------------+
only showing top 10 rows



8. Filter rows where torque_nm is not null.

In [17]:
(
    df.select("brand","model","torque_nm").filter(psf.col("torque_nm").isNotNull()).show(100)
)

+--------+--------------------+---------+
|   brand|               model|torque_nm|
+--------+--------------------+---------+
|  Abarth|    500e Convertible|      235|
|  Abarth|      500e Hatchback|      235|
|  Abarth| 600e Scorpionissima|      345|
|  Abarth|        600e Turismo|      345|
|  Aiways|                  U5|      310|
|  Aiways|                  U6|      315|
|    Alfa|Romeo Junior Elet...|      260|
|    Alfa|Romeo Junior Elet...|      345|
|  Alpine|A290 Electric 180 hp|      285|
|  Alpine|A290 Electric 220 hp|      300|
|    Audi|     A6 Avant e-tron|      435|
|    Audi|A6 Avant e-tron p...|      565|
|    Audi|A6 Avant e-tron q...|      855|
|    Audi| A6 Sportback e-tron|      435|
|    Audi|A6 Sportback e-tr...|      565|
|    Audi|A6 Sportback e-tr...|      855|
|    Audi|Q4 Sportback e-tr...|      310|
|    Audi|Q4 Sportback e-tr...|      545|
|    Audi|Q4 Sportback e-tr...|      679|
|    Audi|Q4 Sportback e-tr...|      679|
|    Audi|        Q4 e-tron 40|   

9. retrieve vehicles where range_km is between 250 and 300.

10. Filter SUVs with more than 4 seats.

In [18]:
(
    df.select("brand","model","car_body_type","seats").filter(psf.col("car_body_type")== "SUV").filter(psf.col("seats")>4).orderBy("seats").show(100)
)

+-------------+--------------------+-------------+-----+
|        brand|               model|car_body_type|seats|
+-------------+--------------------+-------------+-----+
|      Lynk&Co|                  02|          SUV|    5|
|        Volvo|EC40 Single Motor...|          SUV|    5|
|           MG|       S5  EV 49 kWh|          SUV|    5|
|       Abarth|        600e Turismo|          SUV|    5|
|           MG|       S5  EV 64 kWh|          SUV|    5|
|       Aiways|                  U6|          SUV|    5|
|           MG|    ZS EV Long Range|          SUV|    5|
|         Alfa|Romeo Junior Elet...|          SUV|    5|
|           MG|ZS EV Standard Range|          SUV|    5|
|         Audi|Q4 Sportback e-tr...|          SUV|    5|
|     Maserati|     Grecale Folgore|          SUV|    5|
|         Audi|Q4 Sportback e-tr...|          SUV|    5|
|Mercedes-Benz|             EQA 250|          SUV|    5|
|         Audi|        Q4 e-tron 45|          SUV|    5|
|Mercedes-Benz|            EQA 

# Aggreate Questions 


11. Calculate the average efficiency_wh_per_km for each/per brand.

In [19]:
(
    df.groupBy("brand").agg(psf.avg("efficiency_wh_per_km").alias("average")).show(10)
)

+-----------+------------------+
|      brand|           average|
+-----------+------------------+
|  Leapmotor|             151.0|
| Volkswagen|150.08695652173913|
|    Peugeot| 159.3684210526316|
|        NIO|181.08333333333334|
|      Lexus|150.66666666666666|
|   Polestar|156.44444444444446|
|     Jaguar|             208.0|
|   Maserati|             200.0|
|Rolls-Royce|             192.0|
|       Jeep|             140.0|
+-----------+------------------+
only showing top 10 rows



12. Find the maximum top_speed_kmh for each brand.

In [20]:
(
    df.select("brand","top_speed_kmh").groupBy("brand").agg(psf.max("top_speed_kmh").alias("max_speed")).orderBy(psf.col("max_speed").desc()).show(10)
)

+-------------+---------+
|        brand|max_speed|
+-------------+---------+
|     Maserati|      325|
|      Porsche|      305|
|        Tesla|      282|
|        Lucid|      270|
|          Kia|      260|
|      Hyundai|      260|
|        Lotus|      260|
|  Rolls-Royce|      250|
|         Audi|      250|
|Mercedes-Benz|      250|
+-------------+---------+
only showing top 10 rows



13. Count how many distinct car_body_type value exists.

In [21]:
(df.agg(psf.count_distinct("car_body_type").alias("distinct number")).show())

+---------------+
|distinct number|
+---------------+
|              8|
+---------------+



14. Compute the total number of vehicle for each segment.

In [22]:
( 
    df.groupBy('segment').count().show()
)

+-----------------+-----+
|          segment|count|
+-----------------+-----+
|     JB - Compact|   44|
|N - Passenger Van|   47|
|         A - Mini|    3|
|      B - Compact|   29|
|       F - Luxury|   51|
|       JD - Large|   58|
|       C - Medium|   34|
|        D - Large|   28|
|    E - Executive|   30|
|      JF - Luxury|   30|
|       I - Luxury|    1|
|        JA - Mini|    2|
|      JC - Medium|   91|
|       G - Sports|    2|
|   JE - Executive|   28|
+-----------------+-----+



15. Group by drivetrain and calculate the average acceleration_0_100_s .

In [23]:
(
    df.groupBy("drivetrain").agg(psf.avg("acceleration_0_100_s").alias("average")).show()
)

+----------+------------------+
|drivetrain|           average|
+----------+------------------+
|       FWD| 9.730128205128207|
|       AWD|4.5539267015706795|
|       RWD| 6.887022900763362|
+----------+------------------+



# Sorting 

16. Sort the dataframe by range_km in descending order.

In [24]:
(df.select("brand","model","range_km").sort("range_km",ascending=False)).show(10)

+-------------+--------------------+--------+
|        brand|               model|range_km|
+-------------+--------------------+--------+
|Mercedes-Benz|            EQS 450+|     685|
|        Lucid|   Air Grand Touring|     665|
|Mercedes-Benz|      EQS 450 4MATIC|     655|
|Mercedes-Benz|      EQS 500 4MATIC|     640|
|Mercedes-Benz|      EQS 580 4MATIC|     640|
|         Audi|A6 Sportback e-tr...|     610|
|         Audi|A6 Sportback e-tr...|     590|
|Mercedes-Benz|  EQS AMG 53 4MATIC+|     585|
|        Lucid|         Air Touring|     580|
|         Audi|A6 Avant e-tron p...|     575|
+-------------+--------------------+--------+
only showing top 10 rows



17. Sort vehicle by battery_capacity_kwh , then by top_speed_kmh.

In [25]:
# (df.select("brand","model","battery_capacity_kwh","top_speed_kmh").sort("battery_capacity_kwh",ascending=False).sort("top_speed_kmh",ascending=False)).show(10)

(df.select("brand","model","battery_capacity_kwh","top_speed_kmh").orderBy("top_speed_kmh","battery_capacity_kwh").show())

+---------+--------------------+--------------------+-------------+
|    brand|               model|battery_capacity_kwh|top_speed_kmh|
+---------+--------------------+--------------------+-------------+
|    Dacia|  Spring Electric 45|                25.0|          125|
|    Dacia|  Spring Electric 65|                25.0|          125|
|Leapmotor|                 T03|                36.0|          130|
|  Renault| 5 E-Tech 40kWh 95hp|                40.0|          130|
|   Nissan|Townstar EV Passe...|                45.0|          130|
|  Renault|Kangoo Grand E-Te...|                45.0|          130|
|  Citroen|e-SpaceTourer M 5...|                46.3|          130|
|  Citroen|e-SpaceTourer XL ...|                46.3|          130|
|     Opel|Zafira-e Life L2 ...|                46.3|          130|
|     Opel|Zafira-e Life L3 ...|                46.3|          130|
|  Peugeot|e-Traveller L2 50...|                46.3|          130|
|  Peugeot|e-Traveller L3 50...|                

# Handling Null 

18. Count how many null values are present in each column.

In [26]:
(
    df.select([psf.sum(psf.col(c).isNull().cast("int")).alias(c) for c in df.columns])
    .show()
)

+-----+-----+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+-------+---------+--------+---------+-------------+----------+
|brand|model|top_speed_kmh|battery_capacity_kWh|battery_type|number_of_cells|torque_nm|efficiency_wh_per_km|range_km|acceleration_0_100_s|fast_charging_power_kw_dc|fast_charge_port|towing_capacity_kg|cargo_volume_l|seats|drivetrain|segment|length_mm|width_mm|height_mm|car_body_type|source_url|
+-----+-----+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+-------+---------+--------+---------+-------------+----------+
|    0|    1|            0|                   0|           0|            202|        7|                   0|       

19. Drop rows with null values.

In [27]:
df.select(psf.count("*").alias("total_rows")).show()

+----------+
|total_rows|
+----------+
|       478|
+----------+



In [28]:
df_dropped_rows = (
    df.na.drop()
)

df_dropped_rows.select(psf.count("*").alias("total_rows")).show()

+----------+
|total_rows|
+----------+
|       263|
+----------+



20. Fill nulls in number_of_cells with 0

In [29]:
df_filled = (
    df.select("number_of_cells").fillna(0)
)

(df_filled.select("number_of_cells").show(100))

+---------------+
|number_of_cells|
+---------------+
|            192|
|            192|
|            102|
|            102|
|              0|
|              0|
|            102|
|            102|
|            184|
|            184|
|            150|
|            180|
|            180|
|            150|
|            180|
|            180|
|            216|
|            288|
|            288|
|            288|
|            216|
|            288|
|            288|
|            288|
|            150|
|            150|
|            180|
|            180|
|            180|
|            180|
|            180|
|            180|
|            180|
|            180|
|            396|
|            396|
|            396|
|            396|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|              0|
|         

21. Fill nulls in towing_capacity_kg with the mean value of column.


In [30]:
mean_value = (
    df.select(psf.mean("towing_capacity_kg")).first()
)[0]

In [31]:
df_filled_with_mean = (
    df.select("brand","model","towing_capacity_kg")
    .fillna({"towing_capacity_kg":mean_value})
)

df_filled_with_mean.show(150)

+--------+--------------------+------------------+
|   brand|               model|towing_capacity_kg|
+--------+--------------------+------------------+
|  Abarth|    500e Convertible|                 0|
|  Abarth|      500e Hatchback|                 0|
|  Abarth| 600e Scorpionissima|                 0|
|  Abarth|        600e Turismo|                 0|
|  Aiways|                  U5|              1052|
|  Aiways|                  U6|              1052|
|    Alfa|Romeo Junior Elet...|                 0|
|    Alfa|Romeo Junior Elet...|                 0|
|  Alpine|A290 Electric 180 hp|               500|
|  Alpine|A290 Electric 220 hp|               500|
|    Audi|     A6 Avant e-tron|              2100|
|    Audi|A6 Avant e-tron p...|              2100|
|    Audi|A6 Avant e-tron q...|              2100|
|    Audi| A6 Sportback e-tron|              2100|
|    Audi|A6 Sportback e-tr...|              2100|
|    Audi|A6 Sportback e-tr...|              2100|
|    Audi|Q4 Sportback e-tr...|

# Distinct & Duplication

22. Get distinct values from the brand column.

In [32]:
(
    df.select("brand").distinct().show(100)
)

+-------------+
|        brand|
+-------------+
|    Leapmotor|
|   Volkswagen|
|      Peugeot|
|          NIO|
|        Lexus|
|     Polestar|
|       Jaguar|
|     Maserati|
|  Rolls-Royce|
|         Jeep|
|       Lancia|
|          Kia|
|         Mini|
|     Dongfeng|
|        Volvo|
|        Omoda|
|      Hyundai|
|        Honda|
|        CUPRA|
|       Abarth|
|         Audi|
|          BYD|
|     Cadillac|
|      Genesis|
|          KGM|
|Mercedes-Benz|
|      Renault|
|       Hongqi|
|        Tesla|
|       Aiways|
|          BMW|
|         Alfa|
|           MG|
|        Maxus|
|        Smart|
|          GWM|
|        Voyah|
|         Opel|
|        Dacia|
|        XPENG|
|        Zeekr|
|      Porsche|
|       Alpine|
|      Citroen|
|       Elaris|
|      VinFast|
|      firefly|
|           DS|
|        Lotus|
|      Skywell|
|      Lynk&Co|
|        Skoda|
|       Toyota|
|        Lucid|
|       Nissan|
|        Mazda|
|         Fiat|
|         Ford|
|       Subaru|
+-------

23. Drop duplicate rows from the dataframe.

In [33]:
duplicate_rows = (
    df.groupBy(df.columns).count().filter(psf.col("count")>1)
)
duplicate_rows.show(100)

+-----+-----+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+-------+---------+--------+---------+-------------+----------+-----+
|brand|model|top_speed_kmh|battery_capacity_kWh|battery_type|number_of_cells|torque_nm|efficiency_wh_per_km|range_km|acceleration_0_100_s|fast_charging_power_kw_dc|fast_charge_port|towing_capacity_kg|cargo_volume_l|seats|drivetrain|segment|length_mm|width_mm|height_mm|car_body_type|source_url|count|
+-----+-----+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+-------+---------+--------+---------+-------------+----------+-----+
+-----+-----+-------------+--------------------+------------+---------------+---------+----------

# String & Column Operations

24. Add a new column brand_model by concatenating brand and model.

In [34]:
df_with_brand_model = (
    df.withColumn("brand_model",psf.concat_ws(" ","brand","model"))
)

In [35]:
(
    df_with_brand_model.select("brand","model","brand_model").show()
)

+------+--------------------+--------------------+
| brand|               model|         brand_model|
+------+--------------------+--------------------+
|Abarth|    500e Convertible|Abarth 500e Conve...|
|Abarth|      500e Hatchback|Abarth 500e Hatch...|
|Abarth| 600e Scorpionissima|Abarth 600e Scorp...|
|Abarth|        600e Turismo| Abarth 600e Turismo|
|Aiways|                  U5|           Aiways U5|
|Aiways|                  U6|           Aiways U6|
|  Alfa|Romeo Junior Elet...|Alfa Romeo Junior...|
|  Alfa|Romeo Junior Elet...|Alfa Romeo Junior...|
|Alpine|A290 Electric 180 hp|Alpine A290 Elect...|
|Alpine|A290 Electric 220 hp|Alpine A290 Elect...|
|  Audi|     A6 Avant e-tron|Audi A6 Avant e-tron|
|  Audi|A6 Avant e-tron p...|Audi A6 Avant e-t...|
|  Audi|A6 Avant e-tron q...|Audi A6 Avant e-t...|
|  Audi| A6 Sportback e-tron|Audi A6 Sportback...|
|  Audi|A6 Sportback e-tr...|Audi A6 Sportback...|
|  Audi|A6 Sportback e-tr...|Audi A6 Sportback...|
|  Audi|Q4 Sportback e-tr...|Au

# Type casting 

26. Cast battery_capacity_kwn from float to integer.

In [36]:
df_bettery_in_int = (
    df.withColumn("battery_capacity_kwh_int" , psf.col("battery_capacity_kWh").cast("int"))
    .drop("battery_capacity_kWh")
)

df_bettery_in_int.select("*").show()

+------+--------------------+-------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+--------------+---------+--------+---------+--------------+--------------------+------------------------+
| brand|               model|top_speed_kmh|battery_type|number_of_cells|torque_nm|efficiency_wh_per_km|range_km|acceleration_0_100_s|fast_charging_power_kw_dc|fast_charge_port|towing_capacity_kg|cargo_volume_l|seats|drivetrain|       segment|length_mm|width_mm|height_mm| car_body_type|          source_url|battery_capacity_kwh_int|
+------+--------------------+-------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+--------------+---------+--------+---------+--------------+--------------------+------------------------+
|

27. Convert range_km to string.

In [37]:
df_bettery_in_int.printSchema()

root
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- top_speed_kmh: integer (nullable = true)
 |-- battery_type: string (nullable = true)
 |-- number_of_cells: integer (nullable = true)
 |-- torque_nm: integer (nullable = true)
 |-- efficiency_wh_per_km: integer (nullable = true)
 |-- range_km: integer (nullable = true)
 |-- acceleration_0_100_s: double (nullable = true)
 |-- fast_charging_power_kw_dc: integer (nullable = true)
 |-- fast_charge_port: string (nullable = true)
 |-- towing_capacity_kg: integer (nullable = true)
 |-- cargo_volume_l: integer (nullable = true)
 |-- seats: integer (nullable = true)
 |-- drivetrain: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- length_mm: integer (nullable = true)
 |-- width_mm: integer (nullable = true)
 |-- height_mm: integer (nullable = true)
 |-- car_body_type: string (nullable = true)
 |-- source_url: string (nullable = true)
 |-- battery_capacity_kwh_int: integer (nullable = true)



# More Selections

28. Create a column is_fast_charge_supported where fast_charging_power_kw_dc >50;

In [38]:
df_with_new_col = (
    df.withColumn("is_fast_charge_supported",psf.expr("fast_charging_power_kw_dc > 50"))
)

In [39]:
df_with_new_col.select("brand","is_fast_charge_supported","fast_charging_power_kw_dc").show(100)

+--------+------------------------+-------------------------+
|   brand|is_fast_charge_supported|fast_charging_power_kw_dc|
+--------+------------------------+-------------------------+
|  Abarth|                    true|                       67|
|  Abarth|                    true|                       67|
|  Abarth|                    true|                       79|
|  Abarth|                    true|                       79|
|  Aiways|                    true|                       78|
|  Aiways|                    true|                       78|
|    Alfa|                    true|                       85|
|    Alfa|                    true|                       85|
|  Alpine|                    true|                       70|
|  Alpine|                    true|                       70|
|    Audi|                    true|                      150|
|    Audi|                    true|                      200|
|    Audi|                    true|                      200|
|    Aud

29. Filter all rows where car_body_type is "SUV"

In [40]:
df_SUV_only = (
    df.where(psf.expr("car_body_type == 'SUV'"))
)

(df_SUV_only.select("*").show())

+------+--------------------+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+------------+---------+--------+---------+-------------+--------------------+
| brand|               model|top_speed_kmh|battery_capacity_kWh|battery_type|number_of_cells|torque_nm|efficiency_wh_per_km|range_km|acceleration_0_100_s|fast_charging_power_kw_dc|fast_charge_port|towing_capacity_kg|cargo_volume_l|seats|drivetrain|     segment|length_mm|width_mm|height_mm|car_body_type|          source_url|
+------+--------------------+-------------+--------------------+------------+---------------+---------+--------------------+--------+--------------------+-------------------------+----------------+------------------+--------------+-----+----------+------------+---------+--------+---------+-------------+--------------------+
|Abarth| 600e Scorpion

# Dataframe MetaData

30. Show all column name and their datatypes.

In [41]:
df.printSchema()

root
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- top_speed_kmh: integer (nullable = true)
 |-- battery_capacity_kWh: double (nullable = true)
 |-- battery_type: string (nullable = true)
 |-- number_of_cells: integer (nullable = true)
 |-- torque_nm: integer (nullable = true)
 |-- efficiency_wh_per_km: integer (nullable = true)
 |-- range_km: integer (nullable = true)
 |-- acceleration_0_100_s: double (nullable = true)
 |-- fast_charging_power_kw_dc: integer (nullable = true)
 |-- fast_charge_port: string (nullable = true)
 |-- towing_capacity_kg: integer (nullable = true)
 |-- cargo_volume_l: integer (nullable = true)
 |-- seats: integer (nullable = true)
 |-- drivetrain: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- length_mm: integer (nullable = true)
 |-- width_mm: integer (nullable = true)
 |-- height_mm: integer (nullable = true)
 |-- car_body_type: string (nullable = true)
 |-- source_url: string (nullable = true)



31. Count total number of rows in the dataframe.

In [42]:
(
    df.count()
)

478

32. Checl all entries in the distinct drivetrain column .

In [43]:
(
    df.select("drivetrain").distinct().show()
)

+----------+
|drivetrain|
+----------+
|       FWD|
|       AWD|
|       RWD|
+----------+



33. Count the number of vehicle per number of seats.

# Medium Level 

Window function and Advances Aggregations

34. For Each brand , rank vehicle by range_km in descending order.

In [44]:
windowSpec = psw.Window.partitionBy("brand").orderBy(psf.col("range_km").desc())

df_with_range_km = (
    df.withColumn("range_rank",psf.rank().over(windowSpec))
)

(df_with_range_km.select("brand","range_km","range_rank").show(10))

+------+--------+----------+
| brand|range_km|range_rank|
+------+--------+----------+
|Abarth|     280|         1|
|Abarth|     280|         1|
|Abarth|     225|         3|
|Abarth|     225|         3|
|Aiways|     350|         1|
|Aiways|     315|         2|
|  Alfa|     320|         1|
|  Alfa|     310|         2|
|Alpine|     310|         1|
|Alpine|     305|         2|
+------+--------+----------+
only showing top 10 rows



35. Calculatw the average battery_capacity_kwh and standard deviation per car_body_type.

In [45]:
df_with_avg_stddev = (
    df.groupBy("car_body_type")
    .agg(psf.round(psf.mean("battery_capacity_kwh"),2).alias("Average"),
    psf.round(psf.stddev("battery_capacity_kwh"),2).alias("standard deviation"))
)

df_with_avg_stddev.show(10)

+-------------------+-------+------------------+
|      car_body_type|Average|standard deviation|
+-------------------+-------+------------------+
|          Hatchback|  49.74|             14.51|
|                SUV|  76.69|             18.24|
|              Sedan|   86.8|             16.48|
|     Liftback Sedan|  85.52|             12.06|
|Small Passenger Van|  60.01|             14.49|
|          Cabriolet|  58.08|              27.1|
|              Coupe|   92.5|             13.44|
|     Station/Estate|  83.67|             15.85|
+-------------------+-------+------------------+



36. Find the vehicle with the longest range_km for each segment.

In [46]:
(
    df.groupBy("segment").agg(psf.max("range_km").alias("longest_range")).show()
)

+-----------------+-------------+
|          segment|longest_range|
+-----------------+-------------+
|     JB - Compact|          455|
|N - Passenger Van|          370|
|         A - Mini|          225|
|      B - Compact|          330|
|       F - Luxury|          685|
|       JD - Large|          545|
|       C - Medium|          495|
|        D - Large|          565|
|    E - Executive|          555|
|      JF - Luxury|          540|
|       I - Luxury|          465|
|        JA - Mini|          300|
|      JC - Medium|          500|
|       G - Sports|          425|
|   JE - Executive|          610|
+-----------------+-------------+



# Complex Filtering & Conditions

37. Find SUVs with a top speed over 180 km/h and efficiency under 160 WH/km

In [47]:
(
    df
    .select("brand","model","top_speed_kmh","efficiency_wh_per_km")
    .filter(psf.col("car_body_type") == "SUV")
    .filter(psf.col("top_speed_kmh") > 180 )
    .filter(psf.col("efficiency_wh_per_km") < 160)
    .show(10)
)

+-------+--------------------+-------------+--------------------+
|  brand|               model|top_speed_kmh|efficiency_wh_per_km|
+-------+--------------------+-------------+--------------------+
| Abarth| 600e Scorpionissima|          200|                 158|
| Abarth|        600e Turismo|          200|                 158|
|   Audi| Q6 e-tron Sportback|          210|                 139|
|   Audi|Q6 e-tron Sportba...|          210|                 145|
|   Audi|Q6 e-tron Sportba...|          210|                 149|
|     DS|  N°8 AWD Long Range|          190|                 146|
|     DS|             N°8 FWD|          190|                 141|
|     DS|  N°8 FWD Long Range|          190|                 136|
|Genesis|        GV60 Premium|          185|                 143|
|Genesis|          GV60 Sport|          200|                 157|
+-------+--------------------+-------------+--------------------+
only showing top 10 rows



# JSON EXPORT & SCHEMA

38. Write the Dataframe to a json file with inferred schema.

In [48]:
file_path1 = "exported_json"

In [49]:
(
    df.write.mode("overwrite").format("json").save(file_path1)
)

In [50]:
file_path1 = "exported_json/part-00000-1064f394-3e9d-47a3-9ffc-fb6994d18bc3-c000.json"

root
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- top_speed_kmh: integer (nullable = true)
 |-- battery_capacity_kWh: double (nullable = true)
 |-- battery_type: string (nullable = true)
 |-- number_of_cells: integer (nullable = true)
 |-- torque_nm: integer (nullable = true)
 |-- efficiency_wh_per_km: integer (nullable = true)
 |-- range_km: integer (nullable = true)
 |-- acceleration_0_100_s: double (nullable = true)
 |-- fast_charging_power_kw_dc: integer (nullable = true)
 |-- fast_charge_port: string (nullable = true)
 |-- towing_capacity_kg: integer (nullable = true)
 |-- cargo_volume_l: integer (nullable = true)
 |-- seats: integer (nullable = true)
 |-- drivetrain: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- length_mm: integer (nullable = true)
 |-- width_mm: integer (nullable = true)
 |-- height_mm: integer (nullable = true)
 |-- car_body_type: string (nullable = true)
 |-- source_url: string (nullable = true)

Read the data from the json file.

In [51]:
schema = pst.StructType([
    pst.StructField("brand",pst.StringType(),True),
    pst.StructField("model",pst.StringType(),True),
    pst.StructField("top_speed_kmh",pst.IntegerType(),True),
    pst.StructField("battery_capacity_kWh",pst.DoubleType(),True),
    pst.StructField("battery_type",pst.StringType(),True),
    pst.StructField("number_of_cells",pst.IntegerType(),True),
    pst.StructField("torque_nm",pst.IntegerType(),True),
    pst.StructField("efficiency_wh_per_km",pst.IntegerType(),True),
    pst.StructField("range_km",pst.IntegerType(),True),
    pst.StructField("acceleration_0_100_s",pst.DoubleType(),True),
    pst.StructField("fast_charging_power_kw_dc",pst.IntegerType(),True),
    pst.StructField("fast_charge_port",pst.StringType(),True),
    pst.StructField("towing_capacity_kg",pst.IntegerType(),True),
    pst.StructField("cargo_volume_l",pst.IntegerType(),True),
    pst.StructField("seats",pst.IntegerType(),True),
    pst.StructField("drivetrain",pst.StringType(),True),
    pst.StructField("segment",pst.StringType(),True),
    pst.StructField("length_mm",pst.IntegerType(),True),
    pst.StructField("width_mm",pst.IntegerType(),True),
    pst.StructField("height_mm",pst.IntegerType(),True),
    pst.StructField("car_body_type",pst.StringType(),True),
    pst.StructField("source_url",pst.StringType(),True),
])

In [52]:
df_read_from_json = (
    spark.read.schema(schema).json(file_path1)
)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/d:/AllSubjectNotes/WEB_TECH_NOTES/DATA_ENGINEER/DAY4/pyspark_practice/exported_json/part-00000-1064f394-3e9d-47a3-9ffc-fb6994d18bc3-c000.json.

In [ ]:
rows = df_read_from_json.select("brand","model","car_body_type").take(10)

for row in rows:
    print(row.asDict())

{'brand': 'Abarth', 'model': '500e Convertible', 'car_body_type': 'Hatchback'}
{'brand': 'Abarth', 'model': '500e Hatchback', 'car_body_type': 'Hatchback'}
{'brand': 'Abarth', 'model': '600e Scorpionissima', 'car_body_type': 'SUV'}
{'brand': 'Abarth', 'model': '600e Turismo', 'car_body_type': 'SUV'}
{'brand': 'Aiways', 'model': 'U5', 'car_body_type': 'SUV'}
{'brand': 'Aiways', 'model': 'U6', 'car_body_type': 'SUV'}
{'brand': 'Alfa', 'model': 'Romeo Junior Elettrica 54 kWh', 'car_body_type': 'SUV'}
{'brand': 'Alfa', 'model': 'Romeo Junior Elettrica 54 kWh Veloce', 'car_body_type': 'SUV'}
{'brand': 'Alpine', 'model': 'A290 Electric 180 hp', 'car_body_type': 'Hatchback'}
{'brand': 'Alpine', 'model': 'A290 Electric 220 hp', 'car_body_type': 'Hatchback'}


# Performance Optimization

39. Cache the dataframe and perform a group by Operation. measure Execution time before and after caching .

In [ ]:
import time

# without using cache

start = time.time() # 11.21.03
df.groupBy("car_body_type").count().show()

print("Without Cache :" , time.time() - start ) # 11.21.10 - 11.21.03


+-------------------+-----+
|      car_body_type|count|
+-------------------+-----+
|          Hatchback|   57|
|                SUV|  244|
|              Sedan|   63|
|     Liftback Sedan|   33|
|Small Passenger Van|   47|
|          Cabriolet|    5|
|              Coupe|    2|
|     Station/Estate|   27|
+-------------------+-----+

Without Cache : 0.4041483402252197


In [ ]:
# With cache 
df.cache()

start1 = time.time() # 11.21.03
df.groupBy("car_body_type").count().show()

print("With Cache :" , time.time() - start1 ) # 11.21.10 - 11.21.03

+-------------------+-----+
|      car_body_type|count|
+-------------------+-----+
|          Hatchback|   57|
|                SUV|  244|
|              Sedan|   63|
|     Liftback Sedan|   33|
|Small Passenger Van|   47|
|          Cabriolet|    5|
|              Coupe|    2|
|     Station/Estate|   27|
+-------------------+-----+

With Cache : 0.24192261695861816


# Complex Sorting 

40. Sort cars by accelration (ascending), then  y torque(descending) and then by range (descending)

In [ ]:
(
    df.orderBy(
        psf.col("acceleration_0_100_s").asc(),
        psf.col("torque_nm").desc(),
        psf.col("range_km").desc()
    ).select("brand","model","acceleration_0_100_s","torque_nm","range_km").show()
)

+-------------+--------------------+--------------------+---------+--------+
|        brand|               model|acceleration_0_100_s|torque_nm|range_km|
+-------------+--------------------+--------------------+---------+--------+
|Mercedes-Benz|            EQS 450+|                 6.2|      568|     685|
|        Lucid|   Air Grand Touring|                 3.0|     1200|     665|
|Mercedes-Benz|      EQS 450 4MATIC|                 5.7|      800|     655|
|Mercedes-Benz|      EQS 580 4MATIC|                 4.4|      858|     640|
|Mercedes-Benz|      EQS 500 4MATIC|                 4.9|      828|     640|
|         Audi|A6 Sportback e-tr...|                 5.4|      565|     610|
|         Audi|A6 Sportback e-tr...|                 4.7|      855|     590|
|Mercedes-Benz|  EQS AMG 53 4MATIC+|                 3.4|     1020|     585|
|        Lucid|         Air Touring|                 3.2|     null|     580|
|        Tesla|  Model S Dual Motor|                 3.2|     null|     575|

# Nested Columns Creation

41. Create a struct column called performance with top_speed_kmh , acceleration_0_100_s and torque_nm.

In [ ]:
df_with_performance_col = (
    df.withColumn("performance",psf.struct("top_speed_kmh","acceleration_0_100_s","torque_nm"))
    .select("brand","model","performance")
)

df_with_performance_col.show()

+------+--------------------+---------------+
| brand|               model|    performance|
+------+--------------------+---------------+
|Abarth|    500e Convertible|{155, 7.0, 235}|
|Abarth|      500e Hatchback|{155, 7.0, 235}|
|Abarth| 600e Scorpionissima|{200, 5.9, 345}|
|Abarth|        600e Turismo|{200, 6.2, 345}|
|Aiways|                  U5|{150, 7.5, 310}|
|Aiways|                  U6|{160, 7.0, 315}|
|  Alfa|Romeo Junior Elet...|{150, 9.0, 260}|
|  Alfa|Romeo Junior Elet...|{200, 6.0, 345}|
|Alpine|A290 Electric 180 hp|{160, 7.4, 285}|
|Alpine|A290 Electric 220 hp|{170, 6.4, 300}|
|  Audi|     A6 Avant e-tron|{210, 6.0, 435}|
|  Audi|A6 Avant e-tron p...|{210, 5.4, 565}|
|  Audi|A6 Avant e-tron q...|{210, 4.5, 855}|
|  Audi| A6 Sportback e-tron|{210, 6.0, 435}|
|  Audi|A6 Sportback e-tr...|{210, 5.4, 565}|
|  Audi|A6 Sportback e-tr...|{210, 4.7, 855}|
|  Audi|Q4 Sportback e-tr...|{160, 8.1, 310}|
|  Audi|Q4 Sportback e-tr...|{180, 6.7, 545}|
|  Audi|Q4 Sportback e-tr...|{180,